# BART baseline
Initial notebook setup.

# **Part 0.** Google Colab environment set up.

In [1]:
# Reinstall ipython kernel to enable autoreload on latest runtime (by 02/12/2026)
# You'll be prompted to restart the session.
!pip install ipython==8.12.0

In [2]:
!pip -q install transformers datasets sentencepiece accelerate

In [6]:
import torch
import transformers
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import BartTokenizer, BartForConditionalGeneration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(F"Device set to {device}")

torch.manual_seed(0)

Device set to cpu


In [7]:
# Load Natural Questions (NQ) dataset
nq = load_dataset("sentence-transformers/natural-questions", split = "train[:20]")

print(nq)
print(nq.column_names)
print(nq[0])

Dataset({
    features: ['query', 'answer'],
    num_rows: 20
})
['query', 'answer']
{'query': 'when did richmond last play in a preliminary final', 'answer': "Richmond Football Club Richmond began 2017 with 5 straight wins, a feat it had not achieved since 1995. A series of close losses hampered the Tigers throughout the middle of the season, including a 5-point loss to the Western Bulldogs, 2-point loss to Fremantle, and a 3-point loss to the Giants. Richmond ended the season strongly with convincing victories over Fremantle and St Kilda in the final two rounds, elevating the club to 3rd on the ladder. Richmond's first final of the season against the Cats at the MCG attracted a record qualifying final crowd of 95,028; the Tigers won by 51 points. Having advanced to the first preliminary finals for the first time since 2001, Richmond defeated Greater Western Sydney by 36 points in front of a crowd of 94,258 to progress to the Grand Final against Adelaide, their first Grand Final appea

# Now, we're going to add some helper functions to process data

In [9]:
def get_questions(data_line):
  return data_line['question']

def get_answers(data_line):
  answer = data_line['answer']
  if isinstance(answer, list):
    answer = answer[0]
  return answer

# Loading BART baseline

In [18]:
# Load the BART baseline
model_name = "facebook/bart-large"
tokenizer = BartTokenizer.from_pretrained(model_name)

model = BartForConditionalGeneration.from_pretrained(model_name)
model = model.to(device)

print("Model loaded: ", model_name)

Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded:  facebook/bart-large


# Small test for a question from the data set

In [19]:
question = "where does a hamster live in the wild?"

inputs = tokenizer(
    question,
    return_tensors="pt",
    max_length=128,
    truncation=True
)
inputs = inputs.to(device)

with torch.no_grad():
  output = model.generate(
      **inputs,
      max_new_tokens = 16,
      num_beams = 4,
      no_repeat_ngram_size = 2,
  )

  prediction = tokenizer.decode(output[0], skip_special_tokens=True)
  print("Question: ", question)
  print("Prediction: ", prediction)

Question:  question: where does a hamster live in the wild?
Prediction:  question: where does a hamster live in the wild?
